## Final Project: Motion Planning

---

**University**: San Diego State University Georgia  
**Course:** Introduction to Artificial Intelligence (CS450)  
**Instructor:** Dr. Shota Tsiskaridze

---

**Student:** Guram Rekhviashvili <br>
**Program:** Computer Science

---

**Date:** May 11, 2026  

---

## Overview

There is a 2-link robot arm operating in a 2D space with static obstacles. Given any pair of reachable, collision-free start and target points, the planner returns the shortest collision-free path.

More specifically, the arm is composed of a fixed shoulder mount, a single elbow joint, and an end-effector which is the final point of the forearm. The workspace is represented as a bounded region populated with circles and convex polygons.

## §1 Problem Statement

### Formal model

The planning problem is formulated as a search through configuration space.

| Symbol | Definition |
|---|---|
| Robot | 2-link arm with fixed base and **zero-width** links (lengths $L_1$, $L_2$) |
| Workspace $W$ | bounded rectangle $W = [x_{\min}, x_{\max}] \times [y_{\min}, y_{\max}] \subset \mathbb{R}^2$, with hard walls — the arm cannot leave $W$ |
| Obstacles | finite set $\mathcal{O} = \{O_1, \dots, O_M\}$ of convex regions $O_i \subset W$ (circles and polygons) |
| Configuration | $q = (\theta_1, \theta_2) \in C$, where $C = [0, \pi] \times [-\pi, \pi]$ |
| Free C-space | $C_{\text{free}} = \{q \in C : \text{arm}(q) \cap O_i = \emptyset \;\forall i, \text{ and arm}(q) \subset W\}$ |
| Path | $\tau(t), \text{where } t\in[0, 1], \tau(t)\in C_{\text{free}}; \tau(0)=q_{\text{start}}, \tau(1)=q_{\text{goal}}$ |
| Cost | path length $L(\tau) = \int_0^1 \lVert \tau'(t) \rVert\, dt$ |

The base of the arm is mounted at $y = 0$ and the workspace lies entirely in the upper half-plane ($y \ge 0$). With hard walls, $\theta_1 < 0$ would put the elbow below the floor and is therefore always in collision — so $\theta_1$ is restricted to $[0, \pi]$. $\theta_2$ keeps its full $[-\pi, \pi]$ range because link 2 can fold either way around the elbow.


### Search formulation

| Element | Specification |
|---|---|
| State | configuration $q \in C_{\text{free}}$ |
| Initial state | $q_{\text{start}}$ (set by user) |
| Goal state | $q_{\text{goal}}$ (set by user) |
| Actions | infinitesimal moves in $C$; discretized to 8-connected steps on a uniform grid |
| Step cost | Euclidean distance to the neighbor cell |
| Path cost | sum of step costs |

The planner solves
$$\min_\tau L(\tau) \quad \text{s.t.} \quad \tau(0) = q_{\text{start}},\; \tau(1) = q_{\text{goal}},\; \tau(t) \in C_{\text{free}} \;\forall t.$$


### Inputs and outputs

**Inputs:** workspace $W$, obstacle set $\mathcal{O}$, link lengths $(L_1, L_2)$, base position, start point $(x_s, y_s)$, goal point $(x_g, y_g)$, grid resolution $R$.

**Output:** sequence of waypoints $[q_0, q_1, \dots, q_N]$ with $q_0 = q_{\text{start}}$, $q_N = q_{\text{goal}}$, or one of three explicit failure messages.

### Three failure modes

The planner explicitly distinguishes three reasons a query can fail. Each is detected at a distinct stage and reported with a distinct message.

1. **Geometric unreachability** — target $(x, y)$ lies outside the arm's reachable annulus $\{p : |L_1 - L_2| \le \lVert p - \text{base} \rVert \le L_1 + L_2\}$. Detected by `is_point_reachable` before any IK math.
2. **Configuration in collision** — IK succeeds but the resulting cell lies in $C_{\text{obstacle}} = C \setminus C_{\text{free}}$. Detected by indexing the C-space bitmap.
3. **No path** — both endpoints lie in $C_{\text{free}}$ but in different connected components. Detected when `plan_path` returns None after A* exhausts its open set.

### Task Environment
| Characteristic | This problem |
|---|---|
| Observability | **Fully observable** — workspace bounds, obstacle geometry, and the arm's current configuration are all known with certainty |
| Determinism | **Deterministic** — every action has a single, predictable outcome; no randomness in the environment or the agent |
| Episodic/Sequential | **Sequential** — the planner outputs a sequence of waypoints, each depending on the previous; commitments early in the path constrain what's available later |
| Static/Dynamic | **Static** — obstacles do not move during planning |
| Continuous/Discrete  | **Continuous** in principle ($\theta_1 \in [0, \pi]$, $\theta_2 \in [-\pi, \pi]$, infinitely many actions), but **discretized to a finite grid** for the planner |
| Agents | **Single-agent** — no other agents to cooperate or compete with |
| Knowledge | **Known** — dynamics reduce to geometry, and that geometry is given as input |

### Additional Assumptions

- **Zero-thickness arm**: Links are line segments with no width.
- **Rigid links:** No flexion or compression.

### Constraints

- The workspace is bounded by `bounds` $= (x_{\min}, x_{\max}, y_{\min}, y_{\max})$ and the boundary is a hard wall (obstacle).

## §2 Workspace

The workspace is a 2D representation of the robot's operating environment. A 2-link arm is mounted at a fixed base point, with links connected at the elbow joint. Convex obstacles (circles and polygons) represent fixed parts the arm must avoid.

All distances are in arbitrary units. The arm base sits at the origin; the workspace extends from $x \in [-75, 75]$ and $y \in [0, 100]$. The rectangle boundary is always treated as a hard wall — the elbow and end-effector cannot leave $W$.

---

In [ ]:
# --- Colab setup (no-op when running locally) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/motion-planning
except ImportError:
    pass  # running locally

import sys
if 'src' not in sys.path:
    sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
import time
from IPython.display import HTML

In [ ]:
from workspace import Robot, CircleObstacle, PolygonObstacle, Workspace

robot = Robot(base=(0, 0), link_lengths=(35, 35))
obstacles = [
    CircleObstacle(center_point=(38, 72), radius=16.0),
    CircleObstacle(center_point=(-42, 68), radius=12.0),
    CircleObstacle(center_point=(8, 48), radius=7.0),
    PolygonObstacle.rectangle(center=(-22, 38), width=18, height=14, rotation=np.pi / 6),
    PolygonObstacle.regular(center=(58, 28), radius=10.0, num_vertices=5),
    PolygonObstacle.regular(center=(-52, 22), radius=8.0, num_vertices=6, rotation=np.pi / 8),
]
ws = Workspace(robot, obstacles, bounds=(-75, 75, 0, 100))

## §3 Forward Kinematics

Forward kinematics answers: *given joint angles $(\theta_1, \theta_2)$, where does each part of the arm end up?*

$\theta_1$ is the angle of the first link relative to the $x$-axis. $\theta_2$ is the angle of the second link **relative to the first**, so the second link globally points in direction $\theta_1 + \theta_2$. Applying basic trigonometry along each link gives the $(x, y)$ position of the elbow and end-effector:

$$\text{elbow} = \text{base} + L_1 (\cos \theta_1, \sin \theta_1)$$
$$\text{end} = \text{elbow} + L_2 (\cos(\theta_1 + \theta_2), \sin(\theta_1 + \theta_2))$$

The cell below checks a few configurations and draws the arm at one of them.


In [ ]:
from kinematics import forward_kinematics, inverse_kinematics, is_point_reachable
from visualization import draw_workspace

test_configs = [(0, 0), (np.pi/2, 0), (np.pi/4, -np.pi/2)]

for theta1, theta2 in test_configs:
    elbow, end_eff = forward_kinematics(ws.robot, theta1, theta2)
    print(f"θ1={np.degrees(theta1):.0f}°  θ2={np.degrees(theta2):.0f}°  "                  
        f"→  elbow=({elbow[0]:.2f}, {elbow[1]:.2f})  "                                   
        f"end_eff=({end_eff[0]:.2f}, {end_eff[1]:.2f})")

# theta1=60°, theta2=30° — arm points upper-right, clears all obstacles
draw_workspace(ws, theta1=np.pi/3, theta2=np.pi/6)

## §4 Inverse Kinematics

Inverse kinematics answers the reverse question: given a target point $(x, y)$, what joint angles place the end-effector there?

The target must lie within the reachable annulus — the ring of points at distances between $|L_1 - L_2|$ and $L_1 + L_2$ from the base. Inside that annulus there are generally **two solutions**: the elbow can bend left or right to reach the same point. On the annulus boundary (arm fully extended or fully folded) the two solutions coincide and only one is returned.

The solution uses the law of cosines on the triangle (base, elbow, target):

$$\cos \theta_2 = \frac{d^2 - L_1^2 - L_2^2}{2 L_1 L_2}, \quad d = \lVert \text{target} - \text{base}\rVert$$

This gives two values of $\theta_2$ (positive and negative arccos) — the elbow-left and elbow-right solutions. 

Once $\theta_2$ is known, $\theta_1$ is obtained by decomposing the geometry into two angles: the direction to the target, and a correction induced by the second link. First, compute the angle from the base to the target:
$$ \alpha = \operatorname{atan2}(y - y_0,\; x - x_0) $$

This is the **absolute direction** of the line connecting the base to the target point.
Next, compute the internal angle from the triangle formed by the two links:

$$
\beta = \operatorname{atan2}\big( L_2 \sin \theta_2,\; L_1 + L_2 \cos \theta_2 \big)
$$

This term captures how much the second link “pulls” the end-effector away from the direction of the first link.

The first joint angle is then:

$$
\theta_1 = \alpha - \beta
$$

Geometrically, $\alpha$ points directly toward the target, while $\beta$ accounts for the bend at the elbow. Subtracting $\beta$ aligns the first link so that, after applying $\theta_2$, the end-effector lands exactly on the target.


This is also the entry point for **failure mode 1**: if $d \notin [|L_1 - L_2|, L_1 + L_2]$, IK returns the empty list and the planner reports geometric unreachability.


In [ ]:
# target must be within the reachable annulus: dist in [|L1-L2|, L1+L2] = [10, 70]
target = (40, 30)  # dist ≈ 50 — comfortably reachable

solutions = inverse_kinematics(ws.robot, *target)
print(f"Target {target}: {len(solutions)} solution(s)")
for i, (t1, t2) in enumerate(solutions):
    elbow, end_eff = forward_kinematics(ws.robot, t1, t2)
    print(f"  [{i}] θ1={np.degrees(t1):.1f}°  θ2={np.degrees(t2):.1f}°  "
        f"→  end_eff=({end_eff[0]:.2f}, {end_eff[1]:.2f})")
# draw both elbow solutions side by side
fig, axes = plt.subplots(1, len(solutions), figsize=(7 * len(solutions), 7))
if len(solutions) == 1:
    axes = [axes]
labels = ['elbow-right', 'elbow-left']
for ax, (t1, t2), label in zip(axes, solutions, labels):
    draw_workspace(ws, theta1=t1, theta2=t2, ax=ax, title=label)
    ax.plot(*target, 'g*', ms=14, zorder=5, label='target')
    ax.legend()
plt.tight_layout()
plt.show()

# verify unreachable point
far = (200, 200)
print(f"\nTarget {far}: reachable={is_point_reachable(ws.robot, *far)}  solutions={inverse_kinematics(ws.robot, *far)}")

## §5 Configuration Space

Rather than reasoning about collisions directly in the workspace, the planner operates in **C-space** — a 2D grid where each axis is a joint angle. Every cell $(\theta_1, \theta_2)$ in this grid is either **free** (the arm fits without hitting anything) or **in collision** (at least one link intersects an obstacle, or the arm leaves the workspace).

The bitmap covers $\theta_1 \in [0, \pi]$ and $\theta_2 \in [-\pi, \pi]$. It is built by running forward kinematics over the entire grid using vectorized NumPy operations, then testing each link against every obstacle (also vectorized). For $R = 400$ this is $R^2 = 160{,}000$ cells.

This is the entry point for **failure mode 2**: indexing the bitmap at the start or goal cell tells us immediately whether the configuration is in collision.

In [ ]:
from cspace import build_cspace_bitmap
from visualization import draw_cspace

RESOLUTION = 200

t_start = time.time()
bitmap = build_cspace_bitmap(ws, resolution=RESOLUTION)
elapsed = time.time() - t_start

n_collision = bitmap.sum()
n_total = bitmap.size
print(f"Built {bitmap.shape} bitmap in {elapsed:.3f}s")
print(f"Collision cells: {n_collision:,} / {n_total:,}  ({100 * n_collision / n_total:.1f}% of C-space is blocked)")

draw_cspace(bitmap)

## §6 Algorithm Selection & Theoretical Justification

Path finding splits into two sub-problems:

1. **Search-space construction** — turning continuous C-space into a finite graph
2. **Graph search** — finding a shortest path in that graph

### Why Grid A\* fits this problem

1. **The C-space bitmap is already a search graph.** Every free cell is a node; every adjacent free cell is a neighbor. There is no separate construction phase. Sampling-based methods (PRM, RRT) exist to dodge the curse of dimensionality, which simply does not apply at 2 DOF.

2. **The grid is small.** At $R = 200$ we have $R^2 = 40{,}000$ cells; at $R = 400$, $160{,}000$. Bitmap construction is sub-second with vectorized NumPy; A\* search is on the order of tens of milliseconds. $R$ is a tunable parameter.

3. **The Euclidean heuristic is admissible and consistent.** Every grid edge has cost equal to the Euclidean distance between its two cell centers. The heuristic estimates remaining cost as the straight-line Euclidean distance. Since the Euclidean distance is the shortest path theoretically, no actual path can beat that. Thus, this heuristic never overestimates. Consistency follows from the same metric being used everywhere.


## §7 Path Planning with A\*

With the C-space bitmap in hand, path planning reduces to graph search. Each free cell is a node; each pair of 8-connected free cells is an edge with cost equal to the Euclidean distance between their center angles. A\* with an admissible Euclidean heuristic finds the shortest path in $\theta$-space.

The planner snaps start and goal angles to their containing cells, runs A\* on the bitmap, then maps the resulting index sequence back to $(\theta_1, \theta_2)$ cell-center angles. The resulting path is optimal *on the grid*; finer $R$ produces shorter, smoother paths at higher cost.


In [ ]:
from path_planning import plan_path
from visualization import draw_plan

start_config = (0.1, np.pi / 15)
goal_config  = (np.pi / 3, np.pi / 6)

t0 = time.time()
try:
    path = plan_path(bitmap, start_config, goal_config, RESOLUTION)
except ValueError as e:
    print(f"Planner rejected the query: {e}")
    path = None
elapsed = time.time() - t0

if path is None:
    print("No path found — start and goal are in disconnected free regions.")
else:
    thetas = np.array(path)
    diffs  = np.diff(thetas, axis=0)
    path_length = float(np.sum(np.hypot(diffs[:, 0], diffs[:, 1])))
    print(f"Path: {len(path)} waypoints  |  {elapsed * 1000:.1f} ms  |  θ-length = {path_length:.3f} rad")

    draw_plan(ws, bitmap, path, start_config, goal_config)

## §8 Complexity Analysis

| Component | Time | Space | Notes |
|---|---|---|---|
| Bitmap construction | $O(R^2 \cdot M)$ | $O(R^2)$ | $R$ = resolution per axis, $M$ = number of obstacles. Each cell tests two arm segments against $M$ obstacles. |
| Grid A\* (worst) | $O(R^2 \log R^2)$ | $O(R^2)$ | Could touch every free cell; each push/pop on the heap is $O(\log n)$. |
| Grid A\* (average) | $O(L \log L)$ | $O(L)$ | $L$ = number of nodes the heuristic actually expands. With an informative heuristic, $L \ll R^2$ in practice. |


## §9 Animation

The static snapshot above shows start and goal, but the planner produces a sequence of intermediate configurations. The animation below steps through each waypoint: the arm moves in the workspace (left) while a yellow dot traces the trajectory through C-space (right).


In [ ]:
from visualization import animate_path

anim = animate_path(ws, bitmap, path, interval=50)
plt.rcParams['animation.embed_limit'] = 50  # MB
plt.close(anim._fig)
HTML(anim.to_jshtml())

## §10 Interactive Demo (Not supported in Colab)

Click anywhere in the **workspace** panel to set the start position, then click again to set the goal. The planner runs IK on each click (auto-selecting the collision-free elbow solution), finds an A\* path through C-space, and animates the result. All three failure modes are reported in the output if they occur:

- **Geometric unreachability** — click outside the reachable annulus
- **Configuration in collision** — IK lands inside an obstacle
- **No path** — start and goal are valid but disconnected


In [ ]:
try:
    import google.colab
    print("Skipping interactive demo — not supported in Colab. Run locally.")
except ImportError:
    %matplotlib widget
    from visualization import interactive_planner
    interactive_planner(ws, bitmap, RESOLUTION)
